# ADA — Demo del flujo de delegación

Notebook que demuestra el **flujo de delegación** del orquestador multi-agente: una consulta del usuario entra por el **Supervisor**, que enruta dinámicamente entre especialistas hasta generar el informe final.

**Flujo esperado:** `Supervisor → Researcher → Analyst → Validator → Synthesizer → END`.

- El chat model se reemplaza por un **router determinístico** (`FakeChatModel`), igual que en la suite de tests: el notebook no requiere red ni claves de API.
- El corpus técnico y el retriever híbrido (BM25 + embeddings + RRF) son los reales del proyecto.
- Cada celda imprime los eventos reales de LangGraph (`stream_mode="updates"`).

## 1. Preparación del entorno

Agregamos `src/` al path (layout de paquete) e importamos los componentes del orquestador.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))

from ada.decision import build_criteria, extract_decision  # noqa: E402
from ada.graph import compile_graph  # noqa: E402

print("Entorno listo:", PROJECT)

## 2. El router determinístico (simula al Supervisor LLM)

El `FakeChatModel` decide **solo el enrutamiento**, leyendo el mismo contexto que vería el modelo real. Es el mismo enfoque de `tests/unit/test_graph.py`: reemplaza al LLM sin cambiar el grafo.

Regla clave: llegar al **Synthesizer** exige que la validación vigente indique `is_valid=True`. Si la validación falla, el grafo aplica su regla determinista de reparación o finalización segura (`route_after_validator`); el mock nunca envía un resultado no validado al Synthesizer.

In [ ]:
class FakeChatModel:
    """Router determinístico: simula las decisiones del Supervisor."""

    def invoke(self, messages):
        last = messages[-1]
        text = str(last[1] if isinstance(last, tuple) else last)
        if "Research artifact present: False" in text:
            nxt = "researcher"
        elif "Analysis artifact present: False" in text:
            nxt = "analyst"
        elif "Validation result: none" in text:
            nxt = "validator"
        elif "is_valid=True" in text:
            nxt = "synthesizer"
        else:
            nxt = "END"

        class Response:
            content = (
                f'{{"next_agent": "{nxt}", "reason": "demo", '
                '"task_completed": false}'
            )

        return Response()

## 2. La consulta del usuario

Escenario de decisión de arquitectura con alternativas explícitas (monolito modular vs. microservicios) y criterios de evaluación.

In [ ]:
USER_REQUEST = (
    "Decidir si una API .NET debe ser monolito modular o microservicios"
)
print("Solicitud:", USER_REQUEST)

## 3. Estado inicial

La CLI fija la especificación estructurada de la decisión (`decision_spec`) y los criterios con pesos normalizados; los nodos los leen, nunca los inventan.

In [ ]:
decision_spec = extract_decision(USER_REQUEST).model_dump()
requested_criteria = decision_spec["requested_criteria"]
criteria = (
    [c.model_dump() for c in build_criteria(requested_criteria)]
    if requested_criteria
    else []
)

initial_state = {
    "user_request": USER_REQUEST,
    "system_profile": {},
    "criteria": criteria,
    "decision_spec": decision_spec,
    "next_agent": "researcher",
    "routing_reason": "inicio",
    "steps": 0,
    "retries": 0,
    "task_completed": False,
}

print("Criterios:", [c["criterion_id"] for c in criteria])

## 3. Ejecución del grafo con eventos en vivo

Usamos el streaming real de LangGraph. Para cada nodo ejecutado se muestra: el nodo ejecutado, el próximo agente seleccionado, el motivo de la decisión, el resultado de la validación y el estado del informe.

In [ ]:
graph = compile_graph(FakeChatModel(), encoder=None)

final_state: dict = {}
for chunk in graph.stream(initial_state, stream_mode="updates"):
    for node, update in (chunk or {}).items():
        if not isinstance(update, dict):
            continue
        final_state = {**final_state, **update}
        print(f"\n=== Nodo ejecutado: {node} ===")
        if "next_agent" in update:
            print(f"  próximo agente: {update['next_agent']}")
        if "routing_reason" in update:
            print(f"  motivo: {update['routing_reason']}")
        if "validation_result" in update:
            validation = update.get("validation_result")
            if isinstance(validation, dict):
                print(
                    f"  validación: is_valid={validation.get('is_valid')}, "
                    f"errores={len(validation.get('errors', []))}"
                )
            else:
                print("  validación: (sin cambios)")
        if "final_report" in update:
            report = update.get("final_report")
            status = report.get("status") if isinstance(report, dict) else None
            print(f"  informe: {status or 'sin cambios'}")

## 4. Resultado: informe final

El informe solo se genera si el Validator aprobó los artefactos. Verificamos el estado final y la recomendación.

In [ ]:
report = final_state.get("final_report") or {}
print("status:", report.get("status"))
print("recomendación:", report.get("recommendation"))
print("alternativas:", report.get("alternatives_considered"))
print("fuentes citadas:", len(report.get("evidence_ids", [])))

## 3. La traza de contribuciones

El estado compartido registra qué agente contribuyó con qué información (`contributions`), evitando la pérdida de contexto.

In [ ]:
for entry in final_state.get("contributions") or []:
    print(
        f"paso {entry.get('step')}: {entry.get('agent')} — "
        f"{entry.get('artifact_type')} — {entry.get('summary')}"
    )

## Resumen

- El **Supervisor** decidió dinámicamente la secuencia de especialistas a partir del estado compartido.
- **Researcher** recuperó evidencia real del corpus con el retriever híbrido; **Analyst** construyó la comparación y **Validator** verificó los contratos antes de que **Synthesizer** generara el informe.
- La traza `contributions` deja evidencia de la delegación en cada paso.

Para una demo con el LLM en vivo (requiere `GROQ_API_KEY`): `uv run python -m ada "..."`.